# Day 1 Assignment: The Lakehouse, Delta Lake & Notebooks

## Task 1: 
Create a notebook and add a markdown cell summarizing, in your own words, the difference between a warehouse, a lake, and a lakehouse. 

#### 1. Warehouse:
- Data warehouse is a storage unit where we store the structured data like tables and views.
- Data should be clean to be able to store in a warehouse
- Data ready to use and is for fast SQL analytics.

#### 2. Datalake:
- Data Lake is a storage unit where all the unstructred data is stored.
- You can just dump the data in any format here, it is cheap and flexible.
- The lakes on their own will not give the reliablity of the warehouse.

#### 3. lakehouse:
- Lakehouse just trying to get both, store everything cheaply like a lake, but also add a transaction layer on the top so you get a warehouse style reliablity: ACID transactions, schema enforcement time travel.

## Task 2:
Create a Delta table from ~10 rows of sample product data (product_id, name, category, price).

In [0]:
%sql
drop table if exists dev.default.products

In [0]:
%sql
-- First we will create a catalog
create catalog if not exists dev;

In [0]:
%sql
-- Creating the table

Create table if not exists dev.default.products (
    product_id int,
    product_name string,
    category string,
    price decimal(10, 2)
)

In [0]:
%sql
-- Inserting the sample data

Insert into dev.default.products (product_id, product_name, category, price) values
    ( 101, 'Wireless mouse', 'Electronics', 799 ),
    ( 102, 'Cotton T-Shirt', 'Apparel', 499 ),
    ( 103, 'Stainless Steal Water Bottle', 'House-Hold', 650 ),
    ( 104, 'Running Shoes', 'Sports', 2499 ),
    ( 105, 'Bluetooth Speaker', 'Electronics', 1499 ),
    ( 106, 'Notebook', 'Stationary', 99),
    ( 107, 'Ceramic Coffee Mug', 'House-hold', 349 ),
    ( 108, 'Yoga Mat', 'Sports', 899 ),
    ( 109, 'Desk Lamp', 'House-hold', 1199),
    ( 110, 'Backpack', 'Stationary', 1799 )

## task 3:
Run three separate INSERT/UPDATE statements against the table, then use DESCRIBE HISTORY to view the resulting versions. 


In [0]:
%sql
Insert into dev.default.products (product_id, product_name, category, price) values
    ( 111, 'Sunglasses', 'Accessories', 1999 ),
    ( 112, 'Blue jeans', 'Apparel', 1499 )

In [0]:
%sql
Update dev.default.products 
set price = 1.1 * price 
where category = 'Electronics'

In [0]:
%sql
Update dev.default.products
set price = 0.5 * price
where product_id = 112

In [0]:
%sql
-- Checking the history after manipulating the table

DESCRIBE HISTORY dev.default.products

## Task 4:
(Data Analyst) Use SELECT ... VERSION AS OF to query an older version of the table and note what changed between versions. 

In [0]:
%sql
-- Current Table 
Select * from dev.default.products

In [0]:
%sql
Select * from dev.default.products version as of 1

- The changes in the table of before 3 Inserts/Updates:
    1. There are two more products
    2. The price of all electronics is increased by 10% 
    3. The price ofproduct with product Id 112 is reduced by 50%


## Task 5:
Deliberately insert a row with an extra column and observe Delta's schema enforcement rejecting it; then re-insert using mergeSchema and confirm schema evolution succeeded.


In [0]:
%sql
Insert into dev.default.products (product_id, product_name, category, price, review) values (113, 'AC', 'House-hold', 6000, 'Very good')

The above cell rejected the additional column for the review

Now will use merge Schema

In [0]:
data = (113, 'Window AC', 'House-hold', 6000, 'Very Good')
schema = ('product_id', 'product_name', 'category', 'price', 'review')
df = spark.createDataFrame([data], schema)
df.write.mode('append').option("mergeSchema", True).saveAsTable('dev.default.products')

# Can also use Schema Evolution command in sql.

In [0]:
%sql
select * from dev.default.products

## Task 6: 
Use time travel (VERSION AS OF and TIMESTAMP AS OF) to reconstruct the table as it looked before a simulated bad update, then write the RESTORE command that would fix it. 

In [0]:
%sql
describe history dev.default.products

In [0]:
%sql
select * from dev.default.products version as of 5

In [0]:
%sql 
restore table dev.default.products to version as of 5

In [0]:
%sql
select * from dev.default.products

## task 7:
Write a short explanation, aimed at a non-technical stakeholder, of why ACID transactions matter when multiple pipelines write to the same table concurrently. 

ACID properties matter because if there is no ACID properites and multiple pipeline are concurrently writing in the table then this would lead to the corrupt and inconsistent data.

What if two pipeline tries to write in a table at the exact same time this would lead to conficts, so ACID transactions helps here to act as a refree between the pipelines to sort this out

ACID:
- Atomicity: This means all or nothing meaning that if the pipeline fails in the middle we don't have to worry about the half saved data. There will be either 0 % or 100 %

- Consistency: It means all data need to follow your predefined rules and if there is any break then that pipeline is stopped and given a error

- Isolation: Meaning that if even 10 pipelines are writing in the same table at the same time, database makes them wait in an orderly manner, and the pipeline can't see what the other pipelines are altering

- Durablity: Once a pipeline recives the sucessfully confirmed it is set in stones. even if an error occours later then data still remains the same.

## Task 8: 
Write a short design note describing how Cyntexa could replace a nightly batch warehouse load with a lakehouse pipeline, calling out specifically where ACID transactions and time travel reduce operational risk compared to the current warehouse. 

We can replace the batch warehouse load with the lakehouse pipeline by when data comes from different sources we do not wait for the nightly updates and just upload the data continously or in small batches in small intervals. So by this the bronze will have a continous supply of data and we can clear it and when needed can get aggeragtes in the gold according to the business requirements.

ACID transactions ensures that either the changes are completely commited or doesn't get committed at all. the table remain valid before and after the transactions. No teo changes affect each other and no two pipeline are accessing the two tables at the same time. When the changes are commited they are not lost.

Time travel provides us to get back to the previous point of correctness if an error is made or some corrupted data is uploaded.

Overall the lakehouse apporoach provide a safer concurrent processing, easier recovery and better than single nightly warehouse load.

## Task 9:
Simulate two concurrent writers appending to the same Delta table (two notebook cells or jobs), then use DESCRIBE HISTORY to explain how the transaction log resolved the write order and what would happen if the writes conflicted. 

For this I have created two notebooks task_1 and task_2 in this folder only both are updating the price of the product_id 2.

I created a job with these notebooks as the tasks as parrallel. The job was triggered manually so both tasks started at the same time.

Both tasks completed successfully. However the job's insight panel flagged a warning "Avoid concurrent Executed Command on table "dev.default.products"". databricks itself detected the two writes even though no coflicts occured.

Describe History showed that each task was completed within some millisecond difference, but since there were two versions even though they started at the same time, whichever task's commit reach the log first completed first then the othere one giving the version as N and N+1.

this means that at the job level both the tasks are not overlapped and since there was a not a perfect overlap it just gave two different versions rather than a conflict.

If the conflict had happened: Delta uses the concurrency control. if both of them had a perfect overlap then the second transaction to attempt the commit will fail, and will not recieve the version number and will not appear on the screen.

In [0]:
%sql
describe history dev.default.products

##Task 10:
(Data Analyst) Write a one-page comparison memo: list 3 concrete advantages a lakehouse gives analysts over a traditional warehouse, and 1 tradeoff to watch for.

Quick comparision based on what actually changes for analyists:

1. **Can store multiple types of data:** A lakehouse can store multiple types of data like structured, semi-structured, and unstructured data so we can use it for many use cases like where raw data is needed it can provide raw data it AI and ML use cases, also where it needs structured data we can get structed data as well and as it is not transformed before hand you can do according to your need.

2. **Usefull for incremental and streaming pipelines and historical data:** Warehouse depends on batch streaming but by using lakehouse we can do incremental and streaming as it supports both pipelines. Also here analysts gets to work on raw historincal data instead of highly processed or transformed data that is stored in the data warehouse as it follows ETL.

3. **Better Data Reliability and Recovery:** Lakehouse technologies such as Delta Lake provide ACID transactions and features such as Time Travel. Time Travel allows analysts or data engineers to examine previous versions of a table. For example, if a data pipeline accidentally changes or deletes records, an earlier version can be inspected or restored. This makes troubleshooting and validating historical results easier.

**Tradeoff:** The main tradeoff of Lakehouse is complexity as for lakehouse it enforce strong governance and unity catalog features, and also for every query you get different log files and paraquete files and to make sure everything works fine. So it introduce more complexity along with all the benifits it provides over the warehouse and data lake.